In [1]:
def get_gambit_matrix(payoffs):
    rows = len(payoffs)
    cols = len(payoffs[0])
    payoff_list = []
    indices = []
    for j in range(cols):
        for i in range(rows):
            ele = payoffs[i][j]
            if ele == 0:
                indices.append(0)
            else:
                payoff_list.append(("", ele, -ele))
                indices.append(len(payoff_list))
    # print(payoff_list, indices)
    return payoff_list, indices

def convert_json_to_gbt(json) -> str:
    template = """
<?xml version="1.0" encoding="UTF-8"?>
<gambit:document xmlns:gambit="http://gambit.sourceforge.net/" version="0.1">
<colors>
<player id="-1" red="0" green="0" blue="0" />
<player id="0" red="154" green="205" blue="50" />
<player id="1" red="255" green="0" blue="0" />
<player id="2" red="0" green="0" blue="255" />
</colors>
<font size="10" family="74" face="Arial" style="90" weight="400" />
<numbers decimals="4"/>
<game>
<nfgfile>
NFG 1 R "{title}" {{ "{P1}" "{P2}" }}

{{ {{ {P1-strats-str} }}
{{ {P2-strats-str} }}
}}
"{comment}"

{{
{payoff-list-str}
}}
{indices-str}
</nfgfile>
</game>
</gambit:document>
"""

    json['P1-strats-str'] = ' '.join(f'"{strat}"' for strat in json["P1-strats"])
    json['P2-strats-str'] = ' '.join(f'"{strat}"' for strat in json["P2-strats"])
    payoff_list, indices = get_gambit_matrix(json["payoffs"])
    json['payoff-list-str'] = '\n'.join(f'{{ "{payoff[0]}" {payoff[1]}, {payoff[2]} }}' for payoff in payoff_list)
    # print(json['payoff-list-str'])
    json['indices-str'] = ' '.join([str(index) for index in indices])
    gbt = template.format(**json)
    return gbt


In [67]:
input_json = {
    "title": "Azucena db3 (+4) into BT on Reina",
    "P1": "Azucena",
    "P2": "Reina",
    "P1-strats": ["BT.1", "BT.1,2", "BT.1,4", "BT.1+2", "BT.3", "BT.1+3", "BT.4", "BT.2(2)",
        "BT.4,3", "BT.3+4",
        "B (exit)", "qcb", "d1"
    ],
    "P2-strats": ["B", "DB", "112", "DB1", "SSR~b", "SSL~b", "Power Crush", "Hop Kick", "Low Parry"],
    "comment": "Azucena db3 (+4) into BT on Reina. BT4, BT.4,3 requires upstream recalculation on change",
    "payoffs": [
        [-0.22, -76, 10, -5, -23, 10, 10, 10, 0],
        [-23.52, -76, 33, -5, -23, 32, 33, 33, -76],
        [-0.22, -76, 31, -5, -23, 30, 31, 31, 21],
        [-23, 17, 73, 67, 17, 17, -6, 14, 17],
        [23, -34, -24, -6, 23, 23, 41, -64, -43],
        [0, -76, 0, -5, -76, 0, 35, -64, -76],
        [-5.68, 15, -24, -6, 15, 15, -8, 52, 15],
        [0, 38, -24, -6, -75, 15, -8, 12, 38],
        [-43, 36, -24, -6, 36, 36, 13, 44, 36],
        [0, 0, -38, 81, -76, 76, -28, -64, 0],
        [0, 0, 0, 0, 0, 0, 31.83, 31.83, 0],
        [0, 0, 68, 0, 0, 0, 31.83, 68, 0],
        [0, 0, 6, 6, 6, 6, 6, 6, -43]
    ]
}

In [68]:
import pygambit as gbt
import pandas as pd

bt4 = gbt.Game.read_game("test.gbt")
result = gbt.nash.lcp_solve(bt4)
eqm = result.equilibria[0]

In [69]:
payoff = f'{float(eqm.payoff(players[0])):.4f}'
payoff

'1.7466'

In [70]:
bt4

,B,DB,112,DB1,SSR~b,SSL~b,Power Crush,Hop Kick,Low Parry
BT.1,"-0.22,0.22","-76,76","10,-10","-5,5","-23,23","10,-10","10,-10","10,-10","0,0"
"BT.1,2","-23.52,23.52","-76,76","33,-33","-5,5","-23,23","32,-32","33,-33","33,-33","-76,76"
"BT.1,4","-0.22,0.22","-76,76","31,-31","-5,5","-23,23","30,-30","31,-31","31,-31","21,-21"
BT.1+2,"-23,23","17,-17","73,-73","67,-67","17,-17","17,-17","-6,6","14,-14","17,-17"
BT.3,"23,-23","-34,34","-24,24","-6,6","23,-23","23,-23","41,-41","-64,64","-43,43"
BT.1+3,"0,0","-76,76","0,0","-5,5","-76,76","0,0","35,-35","-64,64","-76,76"
BT.4,"-5.68,5.68","15,-15","-24,24","-6,6","15,-15","15,-15","-8,8","52,-52","15,-15"
BT.2(2),"0,0","38,-38","-24,24","-6,6","-75,75","15,-15","-8,8","12,-12","38,-38"
"BT.4,3","-19.85,19.85","36,-36","-24,24","-6,6","36,-36","36,-36","13,-13","44,-44","36,-36"
BT.3+4,"0,0","0,0","-38,38","81,-81","-76,76","76,-76","-28,28","-64,64","0,0"


In [71]:
players = [x.label for x in bt4.players]
players
equilibria = []


for player in players:
    for strategy, dist in eqm[player]:
        if strategy.label in ['BT.4','BT.4,3']:
            dist_formatted = f"{100*float(dist):.2f}%"
            equilibria.append({"player": strategy.player.label,
                               "strategy": strategy.label,
                               "distribution": dist_formatted})

In [72]:
equilibria

[{'player': 'Azucena', 'strategy': 'BT.4', 'distribution': '0.00%'},
 {'player': 'Azucena', 'strategy': 'BT.4,3', 'distribution': '19.35%'}]

In [79]:
import numpy as np

payoffs = np.arange(-24, -18, 0.5).tolist()

for p in payoffs:
    input_json['payoffs'][8][0] = p
    
    print(f"BT.4,3 Payoff: {p}")
    
    f = convert_json_to_gbt(input_json)
    with open("test.gbt", 'w') as output:
        output.write(f)

    bt4 = gbt.Game.read_game("test.gbt")
    result = gbt.nash.lcp_solve(bt4)
    eqm = result.equilibria[0]
    payoff = f'{float(eqm.payoff(players[0])):.4f}'
    print(f"{players[0]} Payoff: {payoff}")
    
    equilibria = []
    for player in players:
        for strategy, dist in eqm[player]:
            if strategy.label in ['BT.4','BT.4,3']:
                dist_formatted = f"{100*float(dist):.2f}%"
                equilibria.append({"player": strategy.player.label,
                                   "strategy": strategy.label,
                                   "distribution": dist_formatted})
    print(equilibria)
    print()

BT.4,3 Payoff: -24.0
Azucena Payoff: 1.6389
[{'player': 'Azucena', 'strategy': 'BT.4', 'distribution': '33.75%'}, {'player': 'Azucena', 'strategy': 'BT.4,3', 'distribution': '0.00%'}]

BT.4,3 Payoff: -23.5
Azucena Payoff: 1.6389
[{'player': 'Azucena', 'strategy': 'BT.4', 'distribution': '33.75%'}, {'player': 'Azucena', 'strategy': 'BT.4,3', 'distribution': '0.00%'}]

BT.4,3 Payoff: -23.0
Azucena Payoff: 1.6389
[{'player': 'Azucena', 'strategy': 'BT.4', 'distribution': '33.75%'}, {'player': 'Azucena', 'strategy': 'BT.4,3', 'distribution': '0.00%'}]

BT.4,3 Payoff: -22.5
Azucena Payoff: 1.6389
[{'player': 'Azucena', 'strategy': 'BT.4', 'distribution': '33.75%'}, {'player': 'Azucena', 'strategy': 'BT.4,3', 'distribution': '0.00%'}]

BT.4,3 Payoff: -22.0
Azucena Payoff: 1.6389
[{'player': 'Azucena', 'strategy': 'BT.4', 'distribution': '33.75%'}, {'player': 'Azucena', 'strategy': 'BT.4,3', 'distribution': '0.00%'}]

BT.4,3 Payoff: -21.5
Azucena Payoff: 1.6389
[{'player': 'Azucena', 'strateg

In [83]:
(.1952/.3375) * .1952

0.11289789629629632

In [85]:
(.1952/.3375) * .3375 + (.1952/.3375) * .1952

0.30809789629629636

In [76]:
equilibria

[{'player': 'Azucena', 'strategy': 'BT.1', 'distribution': '0.00%'},
 {'player': 'Azucena', 'strategy': 'BT.1,2', 'distribution': '0.00%'},
 {'player': 'Azucena', 'strategy': 'BT.1,4', 'distribution': '2.28%'},
 {'player': 'Azucena', 'strategy': 'BT.1+2', 'distribution': '9.04%'},
 {'player': 'Azucena', 'strategy': 'BT.3', 'distribution': '24.52%'},
 {'player': 'Azucena', 'strategy': 'BT.1+3', 'distribution': '0.00%'},
 {'player': 'Azucena', 'strategy': 'BT.4', 'distribution': '33.75%'},
 {'player': 'Azucena', 'strategy': 'BT.2(2)', 'distribution': '13.44%'},
 {'player': 'Azucena', 'strategy': 'BT.4,3', 'distribution': '0.00%'},
 {'player': 'Azucena', 'strategy': 'BT.3+4', 'distribution': '0.00%'},
 {'player': 'Azucena', 'strategy': 'B (exit)', 'distribution': '0.00%'},
 {'player': 'Azucena', 'strategy': 'qcb', 'distribution': '16.98%'},
 {'player': 'Azucena', 'strategy': 'd1', 'distribution': '0.00%'},
 {'player': 'Reina', 'strategy': 'B', 'distribution': '51.87%'},
 {'player': 'Reina

In [ ]:
when the attacker uses BT.4, they enter the subgame BT.4 (-5) on block 51.87% of the time.
In those 51.87% of the time, you should fire 3 off 13% based on the nash equilibrium.


In [77]:
.5187*.13

0.067431

In [80]:
19/34

0.5588235294117647

In [ ]:
doing this misses the damage we get from ripping upstream